# Data Manipulation — Basic
## Data Cleaning & Quality Control (R)

**Philosophy:** This note is pattern-first, not syntax-first. It assumes you know
the tidyverse API. Each section answers: *you have data that looks like X — here
is the full pipeline to fix it.*

---

## Decision Table

| If your data has... | Go to |
| :--- | :--- |
| Unknown shape, types, or quality | §1 — Audit |
| Dates as strings, IDs as doubles, booleans as character | §2 — Fix types |
| Missing values and unsure what to do | §3 — Missing Values |
| Suspected duplicate rows | §4 — Duplicates |
| Inconsistent category labels (`"US"`, `"us"`, `" USA "`) | §5 — String & Categorical |
| Extreme values distorting results | §6 — Outliers |
| Column names with spaces, caps, or special characters | §7 — Column Names |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` **(this note)** | Cleaning & QC — audit, types, nulls, dupes, strings, outliers, column names |
| `DM_Intermediate` | Reshaping & combining — wide/long, group-by, multi-table joins, time-series, nested, many files |
| `DM_Advanced` | Feature engineering — encoding, transforms, binning, date, lag/window, interactions, text, leakage |
| `Reference_Tidyverse / Reference_BaseR` | Syntax lookups for individual functions |

> **Environment:** R 4.x / tidyverse 2.x. dplyr verbs always return a **new** object
> — there is no `inplace=True`-style trap and no copy-on-write subtlety to manage;
> every `df <- df |> mutate(...)` is already explicit and safe. A sample frame is
> defined below so the patterns are runnable.

In [ ]:
library(dplyr)
library(tidyr)
library(stringr)
library(lubridate)

# Sample messy frame to run the cleaning patterns against.
# Replace with your own:  df <- readr::read_csv('data.csv')
df <- tibble::tibble(
  user_id    = c(1, 2, 3, 4, 4, 5),
  event_date = c('2024-01-05','2024-01-06','2024/01/07','08-01-2024','08-01-2024', NA),
  age        = c(34, 28, -5, 41, 41, 130),
  revenue    = c('$1,200', '950', '', '2,000', '2,000', '15'),
  country    = c('US', ' us ', 'USA', 'Canada', 'Canada', 'cnaada'),
  platform   = c('iOS','Android','Web','iOS','iOS','Web')
)
print(df)

---
## §1 — Audit a New Dataset

Run this sequence every time you load a new dataset — before touching anything.
The goal is to build a complete picture of what you have and what needs fixing.

In [ ]:
library(dplyr)
library(readr)

# Using the sample df defined in SS0 above.
# In production: df <- read_csv('your_file.csv')
# The patterns below work on any data frame.

# Shape and preview
cat("dim(df)  :", dim(df), "\n")
head(df, 3)


**What each step is looking for:**

| Step | Red flags to catch |
| :--- | :--- |
| Shape + preview | Extra header rows, merged cells exported from Excel, trailing commas |
| `glimpse()` | Date columns as `character`, ID columns as `double` (signals nulls present), boolean as `chr`/`int` |
| Null summary | Columns with >50% nulls — question if they're usable |
| Duplicates | Full duplicates = data pipeline bug; key-level = fan-out from a join |
| Cardinality | String columns with thousands of unique values — probably IDs, not categories |
| Value ranges | Impossible values: negative age, rate > 100%, future dates in historical data |

**Common mistakes:**
- Skipping the audit and jumping straight to analysis — silent errors propagate and are hard to trace back
- Using `summary()` alone on the full frame — numeric and character columns summarize very differently; always split by type as shown, or pair with `glimpse()`
- Treating a high null rate as always bad — sometimes nulls are meaningful (e.g. `cancellation_date` is `NA` = not cancelled)

**Three more audit checks for §1:** detect columns that secretly mix types
(rarer in R than pandas, since R columns are type-homogeneous by construction —
but still possible via `list`-columns), measure true memory use, and turn ad-hoc
checks into reusable assertions.

In [ ]:
library(dplyr)
s <- tibble::tibble(
  id       = c(1, 2, 3, 4),
  mixed    = list(1, "two", 3, 4.0),                 # list-column holding mixed types -- R's closest analog to pandas' object-mixing
  platform = c('ios','android','ios','web'),
  revenue  = c(10.0, 20.0, NA, 40.0)
)

# 1) Columns that secretly mix types -- in R this mainly shows up in list-columns,
#    since atomic vector columns are coerced to one type automatically on creation
#    (R's biggest structural difference from pandas' object dtype)
mixed_type_cols <- function(df) {
  out <- list()
  for (c in names(df)) {
    if (is.list(df[[c]])) {
      kinds <- unique(sapply(df[[c]][!sapply(df[[c]], is.null)], function(v) class(v)[1]))
      if (length(kinds) > 1) out[[c]] <- kinds
    }
  }
  out
}
cat('mixed-type columns:', paste(names(mixed_type_cols(s)), collapse = ", "), '\n')

# 2) True memory footprint -- object.size() already counts actual data, not just pointers
print(sapply(s, object.size))

# 3) Reusable validation: promote ad-hoc checks to assertions you re-run after each step
validate <- function(df) {
  stopifnot(
    "id must be unique" = !any(duplicated(df$id)),
    "revenue must be non-negative" = all(df$revenue[!is.na(df$revenue)] >= 0),
    "unexpected platform" = all(df$platform %in% c('ios', 'android', 'web'))
  )
  cat('all checks passed\n')
}
validate(s)

---
## §2 — Fix Types

Wrong types are the silent cause of failed joins, incorrect aggregations, and
broken date math. Fix them immediately after the audit — everything downstream
depends on correct types.

In [ ]:
library(lubridate)

# ── Pattern: Date columns loaded as strings ──────────────────────────────────
# The sample df has event_date as character. Parse it:
df_fixed <- df |> mutate(
  event_date_parsed = parse_date_time(event_date, orders = c("ymd", "dmy", "mdy"))
)
cat("Before:", class(df$event_date), "\n")
cat("After :", class(df_fixed$event_date_parsed), "\n")
print(df_fixed |> select(event_date, event_date_parsed))

# ── Pattern: Currency string -> numeric ──────────────────────────────────────
df_fixed <- df_fixed |> mutate(
  revenue_num = as.double(stringr::str_remove_all(revenue, "[$,]") |>
                          stringr::str_trim() |>
                          dplyr::na_if(""))
)
cat("\nRevenue after cleaning:", df_fixed$revenue_num, "\n")

# ── Pattern: Low-cardinality string -> factor ─────────────────────────────────
df_fixed <- df_fixed |> mutate(platform = as.factor(platform))
cat("Platform class:", class(df_fixed$platform), "\n")
cat("Levels:", levels(df_fixed$platform), "\n")


**Type-fix decision tree:**

```
Column shows as <chr> in glimpse()?
+-- Looks like a date                -> lubridate::ymd() / parse_date_time()
+-- Has $, %, commas in values       -> str_remove_all() then as.double()
+-- Has True/False/Yes/No strings    -> case_match() then as.logical()
+-- Few unique values (<50)          -> as.factor()
+-- Should be numeric but has blanks -> na_if("") then as.double()

Numeric column shows as double but should be integer?
+-- Fill or drop NAs first, then as.integer()
```

**Common mistakes:**
- `as.integer(col)` when the column has `NA` — unlike pandas (which raises `ValueError`), R's `as.integer()` silently keeps the `NA` and casts everything else; this is *safer* but means a bad cast can go unnoticed without an explicit check
- Parsing ambiguous date formats like `01/02/03` without specifying the order explicitly via `orders =` — `parse_date_time()` guesses among the orders you give it and can guess wrong if the list is too permissive
- Converting ID columns to `integer`/`double` when they have leading zeros (zip codes, product codes) — use `character` instead, identical trap to pandas

**Timezone handling (a §2 type landmine).** A naive timestamp carries no zone;
mixing tz-aware and tz-naive values in a comparison or join raises an error in R
too. `force_tz()`/`with_tz()` is lubridate's pair for *attaching* vs *converting*
a zone — the same conceptual split as pandas' `tz_localize` vs `tz_convert`.

In [ ]:
library(lubridate)
naive <- ymd_hm(c('2024-06-10 14:30', '2024-06-10 18:30'))
cat('naive tz attribute:', tz(naive), '\n')                      # "UTC" -- R always has SOME tz attribute, but treat this as "naive" until deliberately set
aware <- force_tz(naive, 'America/New_York')                      # attach a zone (naive -> aware), does NOT shift the clock time
cat('tz-aware tz       :', tz(aware), '\n')
cat('converted to UTC  :', format(with_tz(aware, 'UTC')), '\n')   # with_tz CHANGES an already-aware time, shifting the clock

---
## §3 — Handle Missing Values

The syntax for filling/dropping `NA`s is in the Tidyverse reference (§3). This
section focuses on the decision: **which strategy to use and when.**

In [ ]:
library(dplyr)
library(tidyr)

cat("NA summary:\n")
print(colSums(is.na(df)))

# Strategy 1: Drop rows where key columns are NA
df_drop <- df |> drop_na(user_id)
cat(sprintf("\nAfter drop_na(user_id): %d rows (was %d)\n", nrow(df_drop), nrow(df)))

# Strategy 2: Fill with constant
df_fill <- df |> mutate(
  revenue = replace_na(revenue, "0")
)
cat("revenue after replace_na('0'):", df_fill$revenue, "\n")

# Strategy 3: Forward fill (sort first)
df_ffill <- df |> arrange(user_id) |> fill(event_date, .direction = "down")
cat("\nevent_date after forward fill:\n")
print(df_ffill |> select(user_id, event_date))

# Strategy 4: Flag then fill
df_flag <- df |> mutate(
  event_date_missing = as.integer(is.na(event_date)),
  event_date         = replace_na(event_date, "UNKNOWN")
)
cat("\nFlag + fill:\n")
print(df_flag |> select(user_id, event_date, event_date_missing))


**Strategy selection guide:**

| Null rate | Null mechanism | Recommended strategy |
| :--- | :--- | :--- |
| < 5% | Random | Drop the rows |
| Any | Business-defined zero | Fill with 0 or 'Unknown' |
| Any | Time-ordered carry-forward | Forward fill (sort first) |
| Any | Depends on group | Group-conditional fill |
| Any | Missingness is informative | Flag + fill |
| > 60% | Any | Drop the column |

**Common mistakes:**
- Filling with the global mean/median when group values differ significantly — always check if group-conditional fill (§3 Strategy 4) is more appropriate
- Forward-filling without `arrange()`-ing by date first — fills in row order, not time order, producing wrong values, the exact same trap as pandas
- Dropping rows with any `NA` (`drop_na()` with no arguments) when only a few columns matter — pass specific column names to be precise
- Forgetting `na.rm = TRUE` when computing the fill statistic itself (`median()`, `mean()`) — without it, the statistic you're filling with is itself `NA`

---
## §4 — Handle Duplicates

Two distinct types of duplicates — full row duplicates (data pipeline bugs) and
key-level duplicates (fan-out from joins or intentional multi-row data). They
require different fixes.

In [ ]:
library(dplyr)

# Detect
n_full <- sum(duplicated(df))
cat(sprintf("Full duplicates   : %d (%.1f%%)\n", n_full, n_full / nrow(df) * 100))

key <- c("user_id")
n_key <- sum(duplicated(df[key]))
cat(sprintf("Key-level (user_id): %d\n", n_key))

# Inspect duplicated records
dup_mask <- duplicated(df[key]) | duplicated(df[key], fromLast = TRUE)
cat("\nAll copies of duplicated user_ids:\n")
print(df |> filter(dup_mask) |> arrange(user_id))

# Fix: drop full duplicates
df_dedup <- df |> distinct()
cat(sprintf("\nAfter distinct(): %d rows\n", nrow(df_dedup)))

# Fix: keep first per user_id
df_first <- df |> distinct(user_id, .keep_all = TRUE)
cat(sprintf("After distinct(user_id): %d rows\n", nrow(df_first)))

# Fix: aggregate to resolve (sum revenue per user)
df_agg <- df |>
  mutate(revenue_num = as.double(stringr::str_remove_all(revenue, "[$,]") |>
                                  stringr::str_trim() |> dplyr::na_if(""))) |>
  group_by(user_id) |>
  summarise(visit_count = n(), revenue_total = sum(revenue_num, na.rm=TRUE), .groups = "drop")
cat("\nAggregated per user:\n")
print(df_agg)


**Duplicate type decision tree:**

```
Are all columns identical?
+-- Yes -> data pipeline bug -> distinct()
+-- No  -> key-level duplicates -> inspect why
         +-- One should be the "truth" (latest update)  -> arrange + distinct(.keep_all=TRUE)
         +-- All rows have valid data (transactions)     -> aggregate (sum, count, max via summarise)
         +-- Caused by a join                            -> fix the join, not the output
```

**Common mistakes:**
- Dropping duplicates before understanding why they exist — if caused by a bad join, the fix is upstream, not a `distinct()` bandage
- Using `distinct(.keep_all = TRUE)` without `arrange()`-ing first — "first" means first in current row order, which may be arbitrary, same trap as pandas' unsorted `keep='first'`
- Checking only full duplicates and missing key-level duplicates — the more dangerous type is usually the key-level one

---
## §5 — Fix String & Categorical Inconsistencies

The same real-world entity appearing under different labels is one of the most
common data quality issues. `group_by()` and `*_join()` operations silently fail
to match when `"US"` and `"us"` are treated as different groups.

In [ ]:
library(dplyr)
library(stringr)

cat("Raw country values:\n")
print(table(df$country, useNA = "ifany"))

# Fix 1: normalise case and whitespace
df_clean <- df |> mutate(country = str_to_lower(str_trim(country)))
cat("\nAfter lower + trim:\n")
print(table(df_clean$country, useNA = "ifany"))

# Fix 2: map synonyms to canonical form
df_clean <- df_clean |> mutate(country = case_match(country,
  "us"            ~ "United States",
  "usa"           ~ "United States",
  "united states" ~ "United States",
  "canada"        ~ "Canada",
  "cnaada"        ~ "Canada",   # typo fix
  .default = country
))
cat("\nAfter canonical mapping:\n")
print(table(df_clean$country, useNA = "ifany"))

# Fix 3: bucket rare values
top_countries <- df_clean |> count(country, sort=TRUE) |>
  slice_head(n = 2) |> pull(country)
df_clean <- df_clean |> mutate(
  country_grouped = if_else(country %in% top_countries, country, "Other")
)
cat("\nAfter bucketing rare values:\n")
print(table(df_clean$country_grouped))


**Cleaning sequence — always in this order:**

1. `str_trim()` — remove whitespace
2. `str_to_lower()` — normalize case
3. `case_match(..., .default = col)` for typos — fix known typos
4. `case_match(..., .default = col)` for synonyms — unify synonyms (same function, different lookup; R doesn't need two separate verbs)
5. Bucket rare values into `'Other'`
6. Validate against controlled vocabulary

**Common mistakes:**
- Using `case_match()` **without** `.default = col` — unmatched values become `NA`, silently nullifying valid data, the exact same trap as pandas' `.map()` without `.fillna(df['col'])`
- Normalizing case after mapping — map first on lowercased values, not on mixed-case raw values
- Forgetting that `case_match()` matches **exact** values only; `str_detect()`/`str_replace()` use substring/regex matching — they are different tools, same split as pandas' `.replace()` vs `.str.replace()`

---
## §6 — Handle Outliers

Outliers are not always errors. The decision is: is this value impossible,
implausible, or just extreme? Each case requires a different response.

In [ ]:
library(dplyr)

# Parse revenue to numeric for outlier detection
df_num <- df |> mutate(
  revenue_num = as.double(stringr::str_remove_all(revenue, "[$,]") |>
                           stringr::str_trim() |> dplyr::na_if(""))
)

# Method A: IQR
Q1    <- quantile(df_num$revenue_num, 0.25, na.rm = TRUE)
Q3    <- quantile(df_num$revenue_num, 0.75, na.rm = TRUE)
IQR_  <- Q3 - Q1
lower <- Q1 - 1.5 * IQR_
upper <- Q3 + 1.5 * IQR_
outliers_iqr <- df_num |> filter(revenue_num < lower | revenue_num > upper)
cat(sprintf("IQR method: %d outliers  [lower=%.1f, upper=%.1f]\n",
    nrow(outliers_iqr), lower, upper))

# Method B: Z-score
z <- (df_num$revenue_num - mean(df_num$revenue_num, na.rm=TRUE)) /
     sd(df_num$revenue_num, na.rm=TRUE)
cat(sprintf("Z-score method (|z|>2): %d outliers\n", sum(abs(z) > 2, na.rm=TRUE)))

# Treatment: winsorize
p01 <- quantile(df_num$revenue_num, 0.01, na.rm=TRUE)
p99 <- quantile(df_num$revenue_num, 0.99, na.rm=TRUE)
df_num <- df_num |> mutate(revenue_capped = pmin(pmax(revenue_num, p01), p99))
cat(sprintf("Winsorized range: [%.1f, %.1f]\n", min(df_num$revenue_capped, na.rm=TRUE),
    max(df_num$revenue_capped, na.rm=TRUE)))

# Treatment: log transform
df_num <- df_num |> mutate(log_revenue = log1p(revenue_num))
cat("log1p(revenue) range:", round(range(df_num$log_revenue, na.rm=TRUE), 3), "\n")


**Treatment decision guide:**

| Outlier type | Treatment | Reason |
| :--- | :--- | :--- |
| Impossible value (age = -5) | Remove | Data entry error — the row is invalid |
| Plausible but extreme (spend = $1M) | Winsorize or flag | Real event — removing distorts the picture |
| Right-skewed distribution | Log transform | Compresses scale without losing data |
| Extreme but informative (whale user) | Flag + keep | May be the most important segment |

**Common mistakes:**
- Removing outliers reflexively — always ask "is this impossible or just unexpected?"
- Using Z-score on heavily skewed data — Z-score assumes normality; IQR is more robust for skewed distributions
- Winsorizing before splitting train/test — compute the percentile bounds on train set only, then apply to test to avoid leakage
- Using `log()` on columns with zeros — produces `-Inf` with a warning; always use `log1p()`, base R's name for the same function as NumPy's `np.log1p()`

---
## §7 — Clean Column Names

Messy column names break `filter()`/`mutate()` with non-syntactic names (requiring
backtick-quoting everywhere) and SQL integrations. Fix them once, right after
loading. The `janitor` package has a one-call solution; the manual pipeline below
shows what it does internally.

In [ ]:
library(stringr)
library(dplyr)

# Demo with a frame that has messy column names
messy <- tibble::tibble(
  `Total Revenue ($)` = c(100, 200),
  `User ID`           = c(1, 2),
  `Q1 2024 Sales`     = c(50, 80),
  `__internal__`      = c(NA, NA)
)

clean_column_names <- function(df) {
  new_names <- names(df) |>
    str_trim() |>
    str_to_lower() |>
    str_remove_all("[^\\w\\s]") |>
    str_replace_all("\\s+", "_") |>
    str_replace_all("_+", "_") |>
    str_remove("^_+") |> str_remove("_+$")
  names(df) <- new_names
  df
}

cat("Before:\n"); print(names(messy))
cleaned <- clean_column_names(messy)
cat("After:\n");  print(names(cleaned))

# Or use janitor (one line)
# cleaned <- janitor::clean_names(messy)

# Rename specific columns
renamed <- cleaned |> rename(revenue = total_revenue_, user = user_id)
cat("After rename():\n"); print(names(renamed))


**Common mistakes:**
- Cleaning column names in the middle of a pipeline — do it immediately after loading, before any selection or filtering
- Forgetting to handle duplicate column names after cleaning — two different columns silently collapse to the same name (R will actually append `...1`/`...2` automatically on `tibble` creation in some cases, but not after a manual `names<-` assignment like above — always check explicitly)
- Using `str_replace_all(" ", "_")` without collapsing multiple spaces first — `'Total  Revenue'` becomes `'Total__Revenue'`, identical trap to pandas

---
## Decision Guide

```
Just loaded a new dataset?
\-- Always run: dim -> glimpse() -> null summary -> duplicates -> summary() -> value ranges  (§1)

Column type is wrong?
+-- Date stored as string             -> lubridate::ymd() / parse_date_time()           (§2)
+-- ID stored as double               -> fill NAs -> as.integer() or as.character()     (§2)
+-- Number has $, commas              -> str_remove_all() -> as.double()                (§2)
\-- String with <50 unique values     -> as.factor()                                    (§2)

Column has missing values?
+-- Null rate < 5%, random            -> drop_na(col1, col2, ...)                       (§3)
+-- Null means zero by logic          -> replace_na(x, 0)                               (§3)
+-- Null should carry forward         -> arrange first -> group_by() |> fill()          (§3)
+-- Null depends on group             -> group_by() |> mutate(replace_na(...))          (§3)
+-- Null is informative               -> flag column + fill                             (§3)
\-- Null rate > 60%                   -> drop the column                                (§3)

Duplicate rows detected?
+-- All columns identical             -> distinct()                                     (§4)
+-- Same key, need latest record      -> arrange by date -> distinct(.keep_all=TRUE)    (§4)
+-- Same key, all rows are valid      -> group_by() |> summarise()                      (§4)
\-- Caused by a join                  -> fix the join upstream                          (§4)

Category labels are inconsistent?
\-- trim -> lower -> replace typos -> map synonyms -> bucket rare values               (§5)

Outliers detected?
+-- Impossible value                  -> remove the row                                 (§6)
+-- Plausible but extreme             -> pmin/pmax() to winsorize                       (§6)
+-- Right-skewed distribution         -> log1p() transform                              (§6)
\-- Extreme but meaningful            -> flag column and keep                           (§6)

Column names are messy?
\-- trim -> lower -> special chars -> spaces->underscore -> deduplicate                (§7)
```